## Imports and housekeeping

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from numpy.ma.core import indices
from scipy.stats import levene
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from scipy import stats

# Disable these two lines of code during development, they are used to ignore warnings when in "prod".
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def corr_plot(input_df):
    corr_df = input_df.corr()
    corr_mat = corr_df[corr_df.columns[::-1]].stack().reset_index(name="correlation")

    sns.set_style("whitegrid")
    g = sns.relplot(
        data=corr_mat,
        x="level_0", y="level_1", hue="correlation", size="correlation",
        palette="vlag", edgecolor=".7",
        height=10, sizes=(50, 250), hue_norm=(-0.5, 1),size_norm=(-0.5, 1),
    )

    g.set(xlabel="", ylabel="", aspect="equal")
    g.despine(left=True, bottom=True)
    g.ax.margins(.02)
    for label in g.ax.get_xticklabels():
        label.set_rotation(90)

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/LUCE-Blockchain/Databases-for-teaching/refs/heads/main/Framingham%20Dataset.csv')

data

## Research questions
-

## Data exploration and cleanup

In [ ]:
numeric_df = data.select_dtypes(include=['number'])

In [ ]:
data.shape

### Statistics

In [ ]:
data.describe()

### Graphs

In [ ]:
num_features = len(numeric_df.columns)
cols = int(np.ceil(np.sqrt(num_features)))
rows = int(np.ceil(num_features / cols))

# A figure with subplots looks much nicer
fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
axes = axes.flatten()

for i, column in enumerate(numeric_df.columns):
    # Technically not needed but might as well
    numeric_df_nona = numeric_df[column].dropna()

    axes[i].hist(numeric_df_nona, bins=30, alpha=0.7, edgecolor='black')

    if len(numeric_df_nona) > 1:
        density = stats.gaussian_kde(numeric_df_nona)
        xs = np.linspace(numeric_df_nona.min(), numeric_df_nona.max(), 200)
        axes[i].plot(xs, density(xs) * len(numeric_df_nona) * (numeric_df_nona.max() - numeric_df_nona.min()) / 30, 'r-', linewidth=2)

    axes[i].set_xlabel(column)
    axes[i].set_ylabel('Number of Patients')
    axes[i].set_title(f'Distribution of {column}')
    axes[i].grid(axis='y', alpha=0.3)

# Remove any empty subplots if they exist
for j in range(num_features, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
corr_plot(data.dropna())

In [ ]:
binary_cols = [col for col in data.columns if
               data[col].dropna().isin([0, 1, 2, 3, 4]).all() and len(data[col].dropna().unique()) <= 4]

num_binary = len(binary_cols)
cols = int(np.ceil(np.sqrt(num_binary)))
rows = int(np.ceil(num_binary / cols))

fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
axes = axes.flatten() if num_binary > 1 else [axes]

for i, col in enumerate(binary_cols):
    value_counts = data[col].value_counts()
    axes[i].pie(value_counts, labels=value_counts.index, autopct='%1.1f%%', startangle=90)
    axes[i].set_title(f'{col}')

for j in range(num_binary, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
numeric_df_nobin = numeric_df.drop(binary_cols, axis=1)

num_features_no_bin = len(numeric_df_nobin.columns)
cols = int(np.ceil(np.sqrt(num_features_no_bin)))
rows = int(np.ceil(num_features_no_bin / cols))

# A figure with subplots looks much nicer
fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
axes = axes.flatten()

for i, column in enumerate(numeric_df_nobin.columns):
    # Technically not needed but might as well
    numeric_df_nobin_nona = numeric_df_nobin[column].dropna()

    axes[i].hist(numeric_df_nobin_nona, bins=30, alpha=0.7, edgecolor='black')

    if len(numeric_df_nobin_nona) > 1:
        density = stats.gaussian_kde(numeric_df_nobin_nona)
        xs = np.linspace(numeric_df_nobin_nona.min(), numeric_df_nobin_nona.max(), 200)
        axes[i].plot(xs, density(xs) * len(numeric_df_nobin_nona) * (numeric_df_nobin_nona.max() - numeric_df_nobin_nona.min()) / 30, 'r-', linewidth=2)

    axes[i].set_xlabel(column)
    axes[i].set_ylabel('Number of Patients')
    axes[i].set_title(f'Distribution of {column}')
    axes[i].grid(axis='y', alpha=0.3)

# Remove any empty subplots if they exist
for j in range(num_features, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### Detecting outliers in the numerical columns

In [ ]:
numerical_cols = data.select_dtypes(include=np.number).columns
outlier_indices = {}

for col in numerical_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    indices = data[(data[col] < lower_bound) | (data[col] > upper_bound)].index
    if len(indices) > 0:
            outlier_indices[col] = indices.tolist()
            #print(f'{len(indices)} outliers found in {col} column')
    else:
        pass
        #print(f"No outlier data found for column {col}")

outlier_values = []
outliers_count = 0
for col, rows in outlier_indices.items():
    outlier_values.append([col, len(rows), (len(rows) / len(data[col]))])
    outliers_count = outliers_count + 1 if len(rows) > 0 else outliers_count

if outliers_count > 0:
    data_outliers = pd.DataFrame(outlier_values)
    data_outliers.columns = ['Variable', 'Outliers', 'Percentage Outliers']
    s = data_outliers.sort_values(by=['Percentage Outliers'], ascending=False).style.bar(subset=['Percentage Outliers'], color='#d65f5f')
    display(s)
else:
    print('No outlier values found in the dataset.')




### Data imputation

In [ ]:
missing  = 0
misVariables = []
CheckNull = data.isnull().sum()
for var in range(0, len(CheckNull)):
    misVariables.append([data.columns[var], CheckNull[var], round(CheckNull[var]/len(data),3)])
    missing = missing + 1

if missing == 0:
    print('Dataset is complete with no blanks.')
else:
    data_misVariables = pd.DataFrame.from_records(misVariables)
    data_misVariables.columns = ['Variable', 'Missing', 'Percentage missing']
    s = data_misVariables.sort_values(by=['Percentage missing'], ascending=False).style.bar(subset=['Percentage missing'], color='#d65f5f')
    display(s)



Dropping columns with more than 50% missing data

In [ ]:
from imputation_functions import drop_high_missing_cols
data_dropped = drop_high_missing_cols(data, threshold=0.50)

Using KNN Classifier or Regressor (based on data category) on columns between 2% and 50%

In [ ]:
from imputation_functions import knn_impute
data_knn_predicted= knn_impute(data_dropped, min_thresh=0.02, max_thresh=0.50, n_neighbors=5)


Using measures of central tendency on columns where <2% of data is missing

In [ ]:
# The imputation function is defined in imputation_functions.py, but we should import it back into the Jupyter notebook in order to make it easier for the professors to correct.
from imputation_functions import impute_simple_central

data_imputed = impute_simple_central(data_knn_predicted)


Code for showing if the imputation successful was

In [ ]:
missing  = 0
misVariables = []
CheckNull = data_imputed.isnull().sum()
for var in range(0, len(CheckNull)):
    misVariables.append([data_imputed.columns[var], CheckNull[var], round(CheckNull[var]/len(data),3)])
    missing = missing + 1

if missing == 0:
    print('Dataset is complete with no blanks.')
else:
    data_misVariables = pd.DataFrame.from_records(misVariables)
    data_misVariables.columns = ['Variable', 'Missing', 'Percentage missing']
    s = data_misVariables.sort_values(by=['Percentage missing'], ascending=False).style.bar(subset=['Percentage missing'], color='#d65f5f')
    display(s)

In [ ]:
corr_plot(data.drop(columns=['HDLC','LDLC']))
corr_plot(data_imputed)

In [ ]:
for column in data_imputed:
    defined_variable = column
    original = data[defined_variable]
    imputed = data_imputed[defined_variable]

    fig, axes = plt.subplots(1, 2, figsize=(20, 4))

    axes[0].hist(original.dropna(), bins=30, alpha=0.7, color='#5D3A9B', edgecolor='black')
    if len(original.dropna()) > 1:
        from scipy import stats

        density = stats.gaussian_kde(original.dropna())
        xs = np.linspace(original.min(), original.max(), 200)
        axes[0].plot(xs, density(xs) * len(original.dropna()) * (original.max() - original.min()) / 30, 'r-',
                     linewidth=2)
    axes[0].set_xlabel(defined_variable)
    axes[0].set_ylabel('Density')
    axes[0].set_title('Original')
    axes[0].grid(axis='y', alpha=0.3)

    axes[1].hist(imputed, bins=30, alpha=0.7, color='#E66100', edgecolor='black')
    if len(imputed) > 1:
        density = stats.gaussian_kde(imputed)
        xs = np.linspace(imputed.min(), imputed.max(), 200)
        axes[1].plot(xs, density(xs) * len(imputed) * (imputed.max() - imputed.min()) / 30, 'r-', linewidth=2)
    axes[1].set_xlabel(defined_variable)
    axes[1].set_ylabel('Density')
    axes[1].set_title('Imputed')
    axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle(f'Distribution Comparison of {defined_variable}', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


## Data skewness

In [ ]:
# TODO: fix data skewness. Left skewed: use exponential transformation Right skewed: use logarithmic transformation. Use scipy.stats.skew to identify skewness.

## Exploring how the cholesterol differs between Smokers and Non-smokers

In [ ]:
from scipy import stats
mean_cholesterol = data_imputed.groupby('CURSMOKE')['TOTCHOL'].describe()
print("Mean Total Cholesterol:")
print(mean_cholesterol)

nonsmokers_chol = data_imputed[data_imputed['CURSMOKE'] == 0]['TOTCHOL']
smokers_chol = data_imputed[data_imputed['CURSMOKE'] == 1]['TOTCHOL']
_, levene_p = stats.levene(nonsmokers_chol, smokers_chol)

t_stat, p_value = stats.ttest_ind(smokers_chol, nonsmokers_chol,
                                  equal_var=False if levene_p < 0.05 else True)
print("\nIndependent T-Test Results:")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: The difference in mean total cholesterol is statistically significant.\n")
else:
    print("Result: There is no statistically significant difference in mean total cholesterol.\n")

plt.figure(figsize=(8, 6))
sns.boxplot(x='CURSMOKE', y='TOTCHOL', data=data_imputed)

# Improve the plot labels
plt.title('Total Cholesterol by Smoking Status', fontsize=16)
plt.ylabel('Total Cholesterol', fontsize=12)
plt.xlabel('Current Smoker', fontsize=12)
plt.xticks(ticks=[0, 1], labels=['Non-Smoker', 'Smoker'])

plt.show()
plt.close()

## Exploring how age differs for people developed Cardiovascular Disease based on their smoking status?

In [ ]:
patients_with_cvd = data_imputed[data_imputed['CVD'] == 1].copy()
if patients_with_cvd.empty:
    raise ValueError("No individuals with CVD found in the dataset.")

age_cvd_per_smoker_group = patients_with_cvd.groupby('CURSMOKE')['AGE'].describe()
print("Age Statistics for Patients with CVD:")
print(age_cvd_per_smoker_group)
print("\n")

smokers_age_with_cvd = patients_with_cvd[patients_with_cvd['CURSMOKE'] == 1]['AGE']
nonsmokers_age_without_cvd = patients_with_cvd[patients_with_cvd['CURSMOKE'] == 0]['AGE']

if smokers_age_with_cvd.empty or smokers_age_with_cvd.empty:
    raise ValueError("Not enough data for one or both groups to run a t-test.")

_, levene_p = stats.levene(smokers_age_with_cvd, nonsmokers_age_without_cvd)

t_stat, p_value = stats.ttest_ind(smokers_age_with_cvd, nonsmokers_age_without_cvd,
                          equal_var=False if levene_p < 0.05 else True)
print("\nIndependent T-Test Results:")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}\n")
if p_value < 0.05:
    print("Result: Among CVD patients, there is a significant age difference between smokers and non-smokers.\n")
else:
    print("Result: Among CVD patients, there is no significant age difference between smokers and non-smokers.\n")

plt.figure(figsize=(8, 6))

sns.boxplot(x='CURSMOKE', y='AGE', data=patients_with_cvd)

# To see individual data points
sns.stripplot(x='CURSMOKE', y='AGE', data=patients_with_cvd, color=".25", alpha=0.3)

plt.title('Age of Patients with Cardiovascular Disease by Smoking Status', fontsize=16)
plt.ylabel('Age', fontsize=12)
plt.xlabel('Current Smoker', fontsize=12)
plt.xticks(ticks=[0, 1], labels=['Non-Smoker', 'Smoker'])

plt.show()
plt.close()
